> **Kaggle kernel `dr-s4b-k0-quick`.** Quick test of the whole pipeline (toy models, separate folder) + downloads and caches the images. Built by `dr-repro/kaggle/stage4b/build_kernels.py` from `dr-repro/stage4b_push.ipynb`; only storage, resume, sharding, a time guard and the test lock differ. Inputs: none.

# Stage 4b: Closing the Gap to the Paper
### Which training choices move APTOS 2019 grading from our 0.914 towards the paper's 0.954, under a strict protocol?

**Plan and predictions:** [`stage4b_plan.md`](stage4b_plan.md), written before this notebook was run. The model is the Stage 4 ordinal model (EfficientNet-B3, one regression output, MSE, cut-offs tuned on held-out data).

| Part | What | Training | Time on a T4 |
|---|---|---|---|
| 0 | **Protocol gap:** how much optimistic evaluation would add to the Stage 4 numbers | none | seconds |
| 1 | **Cheap levers** at 300 px: lr, augmentation, label cleaning, epochs, pre-processing (2 seeds each) | 12 runs | ~2 h |
| 2 | **Resolution and backbone** with the adopted levers | 4 runs | ~1.5 h |
| 3 | **5-fold cross-validation ensemble** of the 2 best candidates | 10 runs | ~3-5 h |
| final | **Test set, opened once:** every row, headline fixed in advance | none | minutes |

**The rules:** the 550 test images (the same as in Stages 3 and 4) are **locked** until the final section. Every choice is made on validation or out-of-fold predictions by rules written in the code before training. Each run uses its final EMA weights (no best-epoch picking).

**How to run (Colab):** **Runtime -> Change runtime type -> T4 GPU**, then **Run all**. Everything is saved to `MyDrive/dr-grading/stage4b/` after every epoch. **If Colab disconnects or you run out of time, just Run all again later**: finished runs are loaded, the interrupted one resumes. `RUN_BLOCKS` controls which blocks may train; a block that cannot start yet (because the one before it is unfinished) says so and is skipped. Set `QUICK_TEST = True` for a ~10-minute check first.

*This is a reproduction study, not a clinical tool.*

## 1. Setup and settings

In [ ]:
import hashlib
import copy
import io
import json
import math
import os
import random
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, fields, replace
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, train_test_split
from tqdm.auto import tqdm

try:
    import timm
except ImportError:                      # not preinstalled on this machine
    !pip install -q timm
    import timm

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Kaggle: results go to /kaggle/working (this kernel's output). The finished runs and the image cache of the kernels
# this one takes as inputs are copied in first, so the chain of kernels continues where the previous one stopped.
import shutil

WORK_DIR = Path("/kaggle/working/dr-grading")
LOCAL_DIR = Path("/tmp/dr")                                    # fast scratch disk, not part of the output
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
SESSION_START = time.time()
TIME_BUDGET_H = 11.0                                           # Kaggle stops a session at 12 h


def restore_from_inputs():
    # Kernel outputs mount as .../dr-grading/; an uploaded dataset may lose that folder and start at stage4/runs/.
    inputs = Path("/kaggle/input")
    sources = sorted({p for p in inputs.glob("**/dr-grading") if p.is_dir()} |
                     {p.parent.parent for p in inputs.glob("**/stage4/runs") if p.is_dir()})
    # Finished runs first (a finished copy replaces an unfinished one), then every other missing file.
    for src in sources:
        for done in src.glob("*/runs/*/results.json"):
            dst = WORK_DIR / done.parent.relative_to(src)
            if not (dst / "results.json").exists():
                shutil.rmtree(dst, ignore_errors=True)
                shutil.copytree(done.parent, dst, ignore=shutil.ignore_patterns("final_ema.pt"))
    for src in sources:
        for f in src.rglob("*"):
            dst = WORK_DIR / f.relative_to(src)
            if f.is_file() and f.name != "final_ema.pt" and not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
    n_done = len(list(WORK_DIR.glob("*/runs/*/results.json")))
    print(f"restored from {[str(s) for s in sources] or 'no inputs'}: {n_done} finished runs")


restore_from_inputs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found (very slow!)")

In [ ]:
QUICK_TEST = True
RUN_BLOCKS = (1, 2, 3)    # blocks this kernel may train
CHECKPOINT_EVERY = 1      # save a resumable checkpoint every N epochs
SAVE_WEIGHTS = True       # keep each run's final EMA weights on Drive (45-80 MB per run)
# Kaggle kernel k0-quick: Quick test of the whole pipeline (toy models, separate folder) + downloads and caches the images.
B1_SEEDS = None     # Block 1 seeds this kernel trains (None = all)
B2_CANDS = None     # Block 2 candidates this kernel trains (None = all)
B3_TOP = None       # Block 3: ranks in TOP this kernel trains (None = all)
OPEN_TEST = True     # section 11 (the test set) runs only in the last kernel
SPLIT_SEED = 42           # the train/val/test split is the same as in Stages 3 and 4


@dataclass
class Config:
    # R0 = the Stage 4 ordinal model (EfficientNet-B3, paper head with 1 output, MSE, CLAHE, 300 px, Adam lr 1e-4,
    # weight decay 1e-4, batch 16, grad clip 0.1), trained for 20 epochs with warm-up + cosine decay and an EMA.
    backbone: str = "efficientnet_b3"
    image_size: int = 300
    preproc: str = "clahe"        # "clahe" (paper) or "graham" (Ben Graham: subtract the local average colour)
    aug: str = "base"             # "base" = flip + rotation (Stage 3); "strong" = + zoom, shift, brightness, gamma
    epochs: int = 20
    warmup_epochs: int = 1
    batch_size: int = 16
    lr: float = 1e-4              # peak learning rate
    weight_decay: float = 1e-4
    grad_clip: float = 0.1
    ema_decay: float = 0.998
    clean_labels: bool = False    # drop images whose identical copies carry different grades from training
    seed: int = 42                # training seed; the split stays fixed
    pretrained: bool = True
    use_amp: bool = True
    max_train_batches: int = 0    # 0 = all batches; >0 only for the quick test


BASE_CFG = Config()

# Block 1: one change each from R0, 2 seeds. Adopted if the mean gain in cross-fitted val QWK is >= ADOPT_MIN_GAIN.
SCREEN_SEEDS = [42, 43]
LEVERS = {
    "lr3e-4": dict(lr=3e-4),
    "strong_aug": dict(aug="strong"),
    "clean_labels": dict(clean_labels=True),
    "30_epochs": dict(epochs=30),
    "graham": dict(preproc="graham"),         # last: it needs its own pre-processed image cache
}
ADOPT_MIN_GAIN = 0.003

# Block 2: resolution and backbone on top of the adopted levers, 1 seed. The TOP_K best go to Block 3.
CANDIDATES = {
    "b3_300": dict(backbone="efficientnet_b3", image_size=300),
    "b3_384": dict(backbone="efficientnet_b3", image_size=384),
    "b4ns_380": dict(backbone="tf_efficientnet_b4.ns_jft_in1k", image_size=380),
    "v2s21k_384": dict(backbone="tf_efficientnetv2_s.in21k_ft_in1k", image_size=384),
}
TOP_K = 2
CV_FOLDS = 5                                  # Block 3

OUT_DIR = WORK_DIR / "stage4b"
STAGE4_RUNS = WORK_DIR / "stage4" / "runs"    # Stage 4 outputs, for Part 0 and the final comparison
if QUICK_TEST:
    BASE_CFG = replace(BASE_CFG, epochs=2, max_train_batches=3, image_size=96)
    SCREEN_SEEDS = [42]
    LEVERS = {k: LEVERS[k] for k in ("strong_aug", "graham")}
    CANDIDATES = {k: {**v, "image_size": 96 + 32 * i} for i, (k, v) in enumerate(CANDIDATES.items())}
    CV_FOLDS = 2
    OUT_DIR = WORK_DIR / "stage4b_quick_test"                  # never mixed with the real runs
    STAGE4_RUNS = WORK_DIR / "stage4_quick_test" / "runs"
if device.type != "cuda" and not QUICK_TEST:
    raise RuntimeError("This runtime has no GPU (Colab falls back to CPU when the free GPU time runs out). "
                       "Runtime -> Change runtime type -> T4 GPU, then Run all again. On a CPU the settings would "
                       "not match the saved runs, and training would take days.")
if device.type != "cuda":
    BASE_CFG = replace(BASE_CFG, use_amp=False)                 # mixed precision is a GPU feature
RUNS_DIR, FIG_DIR = OUT_DIR / "runs", OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(BASE_CFG), indent=2))

## 2. Data and split

Same source, cache and split as Stages 3 and 4 (stratified 70/15/15, seed 42). Block 3 trains on train + validation (3,112 images) in 5 folds that keep identical images together, so out-of-fold predictions are never made on a copy of a training image.

In [ ]:
HF_REPO = "sngsfydy/aptos"
IMG_DIR = LOCAL_DIR / "aptos512"
DRIVE_CACHE = WORK_DIR / "aptos512.zip"
MAX_SIDE = 512


def list_shards():
    with urllib.request.urlopen(f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main/data") as r:
        return sorted(f["path"] for f in json.load(r) if f["path"].endswith(".parquet"))


def convert_shard(parquet_path, out_dir):
    # Save every image in one parquet shard as a <hash>.jpg (<= 512 px). Returns [(file name, grade), ...].
    rows = []
    pf = pq.ParquetFile(parquet_path)
    for group in range(pf.num_row_groups):
        table = pf.read_row_group(group).to_pydict()
        for img, label in zip(table["image"], table["label"]):
            raw = img["bytes"]
            name = hashlib.sha1(raw).hexdigest()[:16] + ".jpg"     # identical photos get identical names
            im = Image.open(io.BytesIO(raw)).convert("RGB")
            scale = MAX_SIDE / max(im.size)
            im.resize((round(im.width * scale), round(im.height * scale)), Image.LANCZOS).save(out_dir / name, quality=95)
            rows.append((name, int(label)))
    return rows


def download_aptos(out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for shard in list_shards():
        local = LOCAL_DIR / Path(shard).name
        print("downloading", local.name, flush=True)
        subprocess.run(["curl", "-sSL", "--retry", "5", "-o", str(local),
                        f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{shard}"], check=True)
        rows += convert_shard(local, out_dir)
        local.unlink()                                             # free the disk before the next shard
        print(f"  {len(rows)} images so far", flush=True)
    pd.DataFrame(rows, columns=["id_code", "label"]).to_csv(out_dir / "labels.csv", index=False)


if not (IMG_DIR / "labels.csv").exists():
    if DRIVE_CACHE.exists():                                       # Stage 3 (or an earlier session) did the download
        print("restoring cached images ...")
        shutil.unpack_archive(DRIVE_CACHE, IMG_DIR)
    else:
        download_aptos(IMG_DIR)
        print("saving a cache for the next kernels ...")
        shutil.make_archive(str(DRIVE_CACHE.with_suffix("")), "zip", IMG_DIR)

df = pd.read_csv(IMG_DIR / "labels.csv")
df["path"] = df["id_code"].map(lambda name: IMG_DIR / name)
labels = df["label"].to_numpy()
print(f"{len(df)} images, {df['id_code'].nunique()} distinct files")

In [ ]:
def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()


with ThreadPoolExecutor(os.cpu_count()) as pool:
    df["hash"] = list(tqdm(pool.map(file_hash, df["path"]), total=len(df), desc="hashing"))
hashes = df["hash"].to_numpy()
groups = df.groupby("hash")["label"].agg(["size", "nunique"])
CONFLICT = df["hash"].isin(set(groups.index[groups["nunique"] > 1])).to_numpy()


def make_split(positions, seed=SPLIT_SEED, val_frac=0.15, test_frac=0.15):
    # Stratified train/val/test row positions. With all positions this is exactly the Stage 3 / Stage 4 split.
    train_val, test = train_test_split(positions, test_size=test_frac, stratify=labels[positions], random_state=seed)
    train, val = train_test_split(train_val, test_size=val_frac / (1 - test_frac),
                                  stratify=labels[train_val], random_state=seed)
    return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}


SPLIT = make_split(np.arange(len(df)))
TRAINVAL = np.sort(np.concatenate([SPLIT["train"], SPLIT["val"]]))
TEST, Y_TEST = SPLIT["test"], labels[SPLIT["test"]]
# Test images with an identical copy in the training data (train for Blocks 1-2, train + val for Block 3).
TEST_TWIN = np.isin(hashes[TEST], hashes[TRAINVAL])
FOLDS = [(TRAINVAL[a], TRAINVAL[b]) for a, b in
         StratifiedGroupKFold(CV_FOLDS, shuffle=True, random_state=SPLIT_SEED).split(TRAINVAL, labels[TRAINVAL],
                                                                                      groups=hashes[TRAINVAL])]
print(f"train {len(SPLIT['train'])}, val {len(SPLIT['val'])}, test {len(TEST)}; "
      f"{CONFLICT.sum()} images with conflicting duplicate grades; "
      f"{TEST_TWIN.sum()} test images have a twin in train + val")
print("fold sizes (held out):", [len(b) for _, b in FOLDS])

## 3. Metrics and decision rules

Same QWK, cut-off search and metrics as Stage 4. New: **cross-fitted QWK**, the screening metric. The cut-offs are tuned on 4/5 of the held-out images and applied to the other 1/5, in turn, so no image is graded by cut-offs that were tuned on it.

In [ ]:
ROUNDING = np.array([0.5, 1.5, 2.5, 3.5])


def qwk(y, pred, k=NUM_CLASSES):
    # Quadratic weighted kappa from the confusion matrix (same value as sklearn's cohen_kappa_score, much faster).
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k).astype(float)
    weights = np.subtract.outer(np.arange(k), np.arange(k)) ** 2 / (k - 1) ** 2
    expected = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()
    denom = (weights * expected).sum()
    return float(1 - (weights * cm).sum() / denom) if denom > 0 else 0.0


def apply_cutoffs(score, cutoffs):
    return np.digitize(score, cutoffs).astype(int)


def fit_cutoffs(score, y, step=0.01, passes=3):
    # Coordinate search: move one cut-off at a time to the grid value with the best QWK, keeping them ordered.
    cutoffs = ROUNDING.copy()
    best = qwk(y, apply_cutoffs(score, cutoffs))
    for _ in range(passes):
        for i in range(len(cutoffs)):
            lo = cutoffs[i - 1] + step if i > 0 else -0.5
            hi = cutoffs[i + 1] - step if i < len(cutoffs) - 1 else NUM_CLASSES - 0.5
            for value in np.arange(lo, hi + 1e-9, step):
                trial = cutoffs.copy()
                trial[i] = value
                score_trial = qwk(y, apply_cutoffs(score, trial))
                if score_trial > best:                       # strictly better: ties keep the current cut-off
                    best, cutoffs = score_trial, trial
    return cutoffs


def crossfit_qwk(score, y, k=5, seed=0):
    # QWK with tuned cut-offs, but every image is graded by cut-offs tuned on the other images.
    pred = np.empty_like(y)
    for fit, held in StratifiedKFold(k, shuffle=True, random_state=seed).split(score, y):
        pred[held] = apply_cutoffs(score[held], fit_cutoffs(score[fit], y[fit]))
    return qwk(y, pred)


def majority_vote(snapshot_probs):
    # Stage 4 arm A: (n_snapshots, n_images, 5) -> one grade per image (the paper's ensemble rule).
    votes = snapshot_probs.argmax(-1)
    tally = np.stack([(votes == c).sum(0) for c in range(NUM_CLASSES)], 1).astype(float)
    return (tally + 1e-3 * snapshot_probs.mean(0)).argmax(1)


def safe_auc(y_binary, score):
    return float(roc_auc_score(y_binary, score)) if 0 < y_binary.sum() < len(y_binary) else float("nan")


def compute_metrics(y, pred, ref_score):
    # ref_score: any score that ranks "referable DR" (grade >= 2); the regression output itself for our models.
    _, rec, _, _ = precision_recall_fscore_support(y, pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    ref_true, ref_pred = y >= 2, pred >= 2
    return {
        "qwk": qwk(y, pred),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0)),
        **{f"recall_{name}": float(rec[c]) for c, name in enumerate(CLASS_NAMES)},
        "off_by_2_or_more": int(np.sum(np.abs(y - pred) >= 2)),
        "ref_sensitivity": float((ref_pred & ref_true).sum() / max(ref_true.sum(), 1)),
        "ref_specificity": float((~ref_pred & ~ref_true).sum() / max((~ref_true).sum(), 1)),
        "ref_auc": safe_auc(ref_true, ref_score),
    }


def bootstrap_qwk(y, pred_a, pred_b=None, n_boot=2000, seed=0):
    # 95% interval of QWK(pred_a), or of QWK(pred_b) - QWK(pred_a) on the same resampled images.
    rng = np.random.default_rng(seed)
    values = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        values[i] = qwk(y[idx], pred_a[idx]) if pred_b is None else qwk(y[idx], pred_b[idx]) - qwk(y[idx], pred_a[idx])
    return np.percentile(values, [2.5, 97.5])

## 4. Part 0: how much can the evaluation protocol alone add?

The paper does not say how its test images were split off or how the final epoch was chosen. Using the saved Stage 4 outputs (no training), we recompute test QWK under three **optimistic** protocols:

* **model picked on test:** the best of the 11 saved models of a run (best-validation epoch plus 10 snapshots) by test QWK;
* **cut-offs tuned on test:** for the snapshot ensemble;
* **both together.**

These numbers are **never** results. They measure how far a protocol alone can move the score.

In [ ]:
def stage4_outputs(name):
    path = STAGE4_RUNS / name / "outputs.npz"
    return dict(np.load(path)) if path.exists() else None


expected_grade = lambda probs: probs @ np.arange(NUM_CLASSES)
gap_rows = []
for seed in (42, 43, 44):
    ce, mse = stage4_outputs(f"ce_s{seed}"), stage4_outputs(f"mse_s{seed}")
    if ce is None or mse is None:
        continue
    assert np.array_equal(ce["test_idx"], TEST), "Stage 4 used a different test set"
    y, yv = ce["y_test"], ce["y_val"]
    # Cross-entropy (paper-style): honest = arm A (majority vote of the snapshots).
    cands = list(zip([ce["best_test"], *ce["snap_test"]], [ce["best_val"], *ce["snap_val"]]))
    ens = expected_grade(ce["snap_test"].mean(0))
    gap_rows.append({"model": "cross-entropy (arm A)", "seed": seed,
                     "honest": qwk(y, majority_vote(ce["snap_test"])),
                     "model picked on test": max(qwk(y, t.argmax(1)) for t, _ in cands),
                     "cut-offs tuned on test": qwk(y, apply_cutoffs(ens, fit_cutoffs(ens, y))),
                     "both": max(qwk(y, apply_cutoffs(s, fit_cutoffs(s, y)))
                                 for s in [ens] + [expected_grade(t) for t, _ in cands])})
    # Ordinal (arm D): honest = ensemble score with cut-offs tuned on validation.
    cands = list(zip([mse["best_test"][:, 0], *mse["snap_test"][..., 0]], [mse["best_val"][:, 0], *mse["snap_val"][..., 0]]))
    ens_t, ens_v = mse["snap_test"].mean(0)[:, 0], mse["snap_val"].mean(0)[:, 0]
    gap_rows.append({"model": "ordinal (arm D)", "seed": seed,
                     "honest": qwk(y, apply_cutoffs(ens_t, fit_cutoffs(ens_v, yv))),
                     "model picked on test": max(qwk(y, apply_cutoffs(t, fit_cutoffs(v, yv))) for t, v in cands),
                     "cut-offs tuned on test": qwk(y, apply_cutoffs(ens_t, fit_cutoffs(ens_t, y))),
                     "both": max(qwk(y, apply_cutoffs(s, fit_cutoffs(s, y))) for s in [ens_t] + [t for t, _ in cands])})

if gap_rows:
    protocol_gap = pd.DataFrame(gap_rows)
    protocol_gap.to_csv(OUT_DIR / "protocol_gap.csv", index=False)
    print("Test QWK of the Stage 4 runs under honest and optimistic protocols (mean over seeds):")
    display(protocol_gap.groupby("model")[["honest", "model picked on test", "cut-offs tuned on test", "both"]].mean().round(3))
else:
    print(f"No Stage 4 outputs found in {STAGE4_RUNS}; Part 0 skipped.")

## 5. Pre-processing

* **`clahe`** (paper, Stages 3-4): CLAHE on the LAB lightness channel, 5x5 Gaussian blur, square resize.
* **`graham`** (Ben Graham, Kaggle 2015 winner): resize, then subtract a large Gaussian blur (`4*img - 4*blur + 128`), which removes lighting and camera colour differences and keeps local detail. The black background is restored, with the bright rim at its edge trimmed.

One pre-processed copy of all images is kept in memory and rebuilt only when a run needs another size or method.

In [ ]:
def load_shrunk(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    h, w = bgr.shape[:2]
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        bgr = cv2.resize(bgr, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    return bgr


def preprocess(path, size, method):
    # Returns RGB uint8 (size, size, 3).
    bgr = load_shrunk(path)
    if method == "clahe":
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
        bgr = cv2.GaussianBlur(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), (5, 5), 0)
        bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    elif method == "graham":
        bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
        retina = (cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY) > 10).astype(np.uint8)
        retina = cv2.erode(retina, np.ones((max(3, size // 50),) * 2, np.uint8)).astype(bool)
        bgr = cv2.addWeighted(bgr, 4, cv2.GaussianBlur(bgr, (0, 0), size / 30), -4, 128)
        bgr[~retina] = 0
    else:
        raise ValueError(f"unknown pre-processing {method!r}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


images, _images_key = None, None


def ensure_images(cfg):
    # Make the global `images` hold every image pre-processed as this run needs.
    global images, _images_key
    key = (cfg.preproc, cfg.image_size)
    if key == _images_key:
        return
    images = None                                                    # free the old copy first
    paths, out = df["path"].tolist(), np.empty((len(df), cfg.image_size, cfg.image_size, 3), dtype=np.uint8)

    def fill(i):
        out[i] = preprocess(paths[i], cfg.image_size, cfg.preproc)

    t0 = time.time()
    with ThreadPoolExecutor(os.cpu_count()) as pool:
        list(tqdm(pool.map(fill, range(len(paths))), total=len(paths), desc=f"pre-processing {key}", leave=False))
    images, _images_key = out, key
    print(f"  images: {cfg.preproc}, {cfg.image_size} px, {out.nbytes / 1e9:.2f} GB, {time.time() - t0:.0f}s")

## 6. Model and augmentation

Any timm backbone with the paper's head ending in **one output** (the Stage 4 ordinal model). Inputs are normalised with each backbone's own pre-training statistics (the ImageNet-21k EfficientNetV2 expects 0.5/0.5).

**Augmentation** (on the GPU): `base` = random flip + rotation (Stage 3). `strong` adds zoom 0.9-1.15x, a shift of up to 5%, brightness x0.8-1.2 and gamma 0.8-1.25; all of them keep the black background black. **TTA** averages 8 exact views: 4 rotations by 90 degrees, with and without a flip.

In [ ]:
class DRNet(nn.Module):
    def __init__(self, backbone, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512), nn.Dropout(0.5), nn.ReLU(inplace=True),
            nn.Linear(512, 512), nn.Dropout(0.25), nn.ReLU(inplace=True),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


NORM = {}


def set_norm(model):
    # Use the backbone's own pre-training mean/std for every input from now on.
    cfg = model.backbone.pretrained_cfg
    NORM["mean"] = torch.tensor(cfg.get("mean", IMAGENET_MEAN), device=device).view(1, 3, 1, 1)
    NORM["std"] = torch.tensor(cfg.get("std", IMAGENET_STD), device=device).view(1, 3, 1, 1)
    NORM["black"] = -NORM["mean"] / NORM["std"]                     # what a black pixel looks like after normalisation


def to_model_input(batch_uint8):
    # numpy (B, H, W, 3) uint8 -> normalised float tensor (B, 3, H, W) on the GPU.
    x = torch.from_numpy(batch_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return ((x - NORM["mean"]) / NORM["std"]).contiguous(memory_format=torch.channels_last)


def augment_batch(x, strong):
    n = x.shape[0]
    x = x - NORM["black"]                                           # black = 0, so empty corners stay black
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)
    angle = torch.rand(n, device=x.device) * 2 * math.pi
    zoom = (torch.rand(n, device=x.device) * 0.25 + 0.9) if strong else torch.ones(n, device=x.device)
    shift = ((torch.rand(n, 2, device=x.device) - 0.5) * 0.1) if strong else torch.zeros(n, 2, device=x.device)
    cos, sin = angle.cos() / zoom, angle.sin() / zoom
    theta = torch.stack([torch.stack([cos, -sin, shift[:, 0]], 1), torch.stack([sin, cos, shift[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False) + NORM["black"]
    if strong:                                                      # brightness and gamma in pixel space
        pixels = (x * NORM["std"] + NORM["mean"]).clamp(0, 1)
        gain = torch.rand(n, 1, 1, 1, device=x.device) * 0.4 + 0.8
        gamma = torch.exp((torch.rand(n, 1, 1, 1, device=x.device) - 0.5) * 2 * math.log(1.25))
        x = ((pixels ** gamma * gain).clamp(0, 1) - NORM["mean"]) / NORM["std"]
    return x.contiguous(memory_format=torch.channels_last)


@torch.no_grad()
def predict(model, indices, cfg, tta=False, batch_size=32):
    # Regression output per image; with tta, the mean over 4 rotations x {no flip, flip}.
    model.eval()
    out = []
    for start in range(0, len(indices), batch_size):
        x = to_model_input(images[indices[start:start + batch_size]])
        views = [torch.rot90(v, k, dims=(2, 3)).contiguous(memory_format=torch.channels_last)
                 for v in (x, x.flip(-1)) for k in range(4)] if tta else [x]
        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            out.append(torch.stack([model(v).float()[:, 0] for v in views]).mean(0).cpu())
    return torch.cat(out).numpy()

## 7. Training

Same loop as Stage 4, with three changes in R0: **20 epochs**, a **1-epoch linear warm-up then cosine decay** of the learning rate (per step), and an **exponential moving average (EMA)** of the weights, which is what gets evaluated. After the last epoch the EMA model predicts the held-out images (with and without TTA) and the test images. The test predictions are saved but not looked at until the final section.

**Safety net:** a resumable checkpoint every epoch, a live `log.csv`, and `results.json` written last. Re-running skips finished runs and resumes an interrupted one.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def lr_factor(step, cfg, steps_per_epoch):
    warm, total = cfg.warmup_epochs * steps_per_epoch, cfg.epochs * steps_per_epoch
    if step < warm:
        return 0.1 + 0.9 * step / warm
    return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))


@torch.no_grad()
def ema_update(ema, model, step, decay):
    d = min(decay, (1 + step) / (10 + step))                        # shorter memory early in training
    for e, m in zip(ema.state_dict().values(), model.state_dict().values()):
        if e.dtype.is_floating_point:
            e.mul_(d).add_(m.detach(), alpha=1 - d)
        else:
            e.copy_(m)


def train_one_epoch(model, ema, optimizer, scaler, cfg, train_idx, rng, step, steps_per_epoch):
    model.train()
    order = rng.permutation(train_idx)
    total_loss, seen = 0.0, 0
    progress = tqdm(range(steps_per_epoch), desc="  training", leave=False)
    for b in progress:
        for group in optimizer.param_groups:
            group["lr"] = cfg.lr * lr_factor(step, cfg, steps_per_epoch)
        idx = np.sort(order[b * cfg.batch_size:(b + 1) * cfg.batch_size])      # sorted = faster memory reads
        x = augment_batch(to_model_input(images[idx]), strong=cfg.aug == "strong")
        y = torch.from_numpy(labels[idx]).to(device).float()
        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            out = model(x)
        loss = F.mse_loss(out.float()[:, 0], y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                             # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        ema_update(ema, model, step, cfg.ema_decay)
        step += 1

        total_loss += loss.item() * len(y)
        seen += len(y)
        progress.set_postfix(loss=f"{total_loss / seen:.3f}")
    return total_loss / seen, step


def save_atomically(path, obj):
    # Write to a temp file, then rename: a crash mid-save can never leave a broken file behind.
    tmp = path.with_suffix(".tmp")
    try:
        torch.save(obj, tmp)
        tmp.replace(path)
    except (RuntimeError, OSError) as err:
        tmp.unlink(missing_ok=True)
        print(f"  warning: could not save {path.name} ({err})")


def load_run(run_dir, cfg):
    saved = np.load(run_dir / "outputs.npz")
    return {"config": cfg, "log": pd.read_csv(run_dir / "log.csv"), **{k: saved[k] for k in saved.files}}


def run_experiment(name, cfg, train_idx, eval_sets, allowed=True):
    # Train one configuration and save its predictions for every set in eval_sets (the first one is monitored).
    # Returns the run, or None if it is unfinished and allowed=False. Resumes if interrupted.
    run_dir = RUNS_DIR / name
    config_file, done_file, ckpt_path = run_dir / "config.json", run_dir / "results.json", run_dir / "last.pt"
    if cfg.clean_labels:
        train_idx = train_idx[~CONFLICT[train_idx]]
    identity = {**asdict(cfg), "train_md5": hashlib.md5(np.asarray(train_idx).tobytes()).hexdigest(),
                "eval_sets": {k: hashlib.md5(np.asarray(v).tobytes()).hexdigest() for k, v in eval_sets.items()}}
    saved = json.loads(config_file.read_text()) if config_file.exists() else None
    if saved == identity and done_file.exists():                           # finished in an earlier session
        return load_run(run_dir, cfg)
    if saved is not None and done_file.exists():
        # Never throw away a finished run because a setting differs (e.g. a CPU runtime turns AMP off).
        changed = {k: (saved.get(k), v) for k, v in identity.items() if saved.get(k) != v}
        raise RuntimeError(f"{name} already finished with other settings {changed} (saved, now). "
                           f"Delete {run_dir} by hand if you really want to retrain it.")
    if not allowed:
        return None
    if time.time() - SESSION_START > TIME_BUDGET_H * 3600:
        print(f"time guard: not starting {name} ({TIME_BUDGET_H} h used); "
              "push this kernel again to continue")
        return None
    run_dir.mkdir(parents=True, exist_ok=True)
    if saved != identity:                     # a new run, or its settings changed: ignore any old files
        ckpt_path.unlink(missing_ok=True)
        done_file.unlink(missing_ok=True)
        config_file.write_text(json.dumps(identity, indent=2))

    ensure_images(cfg)
    print(f"\n=== {name} ===  {cfg.backbone} {cfg.image_size}px {cfg.preproc} aug={cfg.aug} lr={cfg.lr} "
          f"epochs={cfg.epochs} clean={cfg.clean_labels} seed={cfg.seed} train={len(train_idx)}")
    seed_everything(cfg.seed)
    model = DRNet(cfg.backbone, cfg.pretrained).to(device).to(memory_format=torch.channels_last)
    set_norm(model)
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters():
        p.requires_grad_(False)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    rng = np.random.default_rng(cfg.seed)
    steps_per_epoch = len(train_idx) // cfg.batch_size                     # drop the last partial batch
    if cfg.max_train_batches:
        steps_per_epoch = min(steps_per_epoch, cfg.max_train_batches)
    state, first_epoch = {"log": [], "step": 0, "seconds": 0.0}, 1

    if ckpt_path.exists():                                                 # continue an interrupted run
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        except Exception as err:
            ckpt = None
            print(f"  could not read {ckpt_path.name} ({err}); starting from scratch")
        if ckpt is not None:
            model.load_state_dict(ckpt["model"])
            ema.load_state_dict(ckpt["ema"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scaler.load_state_dict(ckpt["scaler"])
            rng.bit_generator.state = ckpt["rng_numpy"]
            torch.set_rng_state(ckpt["rng_torch"])
            if ckpt["rng_cuda"] is not None:
                torch.cuda.set_rng_state_all(ckpt["rng_cuda"])
            state, first_epoch = ckpt["state"], ckpt["epoch"] + 1
            print(f"  RESUMED after epoch {ckpt['epoch']}")

    monitor_name, monitor_idx = next(iter(eval_sets.items()))
    y_monitor = labels[monitor_idx]
    for epoch in range(first_epoch, cfg.epochs + 1):
        t0 = time.time()
        train_loss, state["step"] = train_one_epoch(model, ema, optimizer, scaler, cfg, train_idx, rng,
                                                    state["step"], steps_per_epoch)
        score = predict(ema, monitor_idx, cfg)
        mon_qwk = qwk(y_monitor, apply_cutoffs(score, ROUNDING))
        state["seconds"] += time.time() - t0
        state["log"].append({"epoch": epoch, "train_loss": train_loss, f"{monitor_name}_qwk_rounded": mon_qwk,
                             f"{monitor_name}_mse": float(np.mean((score - y_monitor) ** 2))})
        pd.DataFrame(state["log"]).to_csv(run_dir / "log.csv", index=False)    # live log, readable at any time
        if epoch % CHECKPOINT_EVERY == 0 and epoch < cfg.epochs:
            save_atomically(ckpt_path, {
                "epoch": epoch, "state": state, "model": model.state_dict(), "ema": ema.state_dict(),
                "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "rng_numpy": rng.bit_generator.state, "rng_torch": torch.get_rng_state(),
                "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})
        eta_min = state["seconds"] / epoch * (cfg.epochs - epoch) / 60
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | loss {train_loss:.3f} | EMA {monitor_name} QWK (rounded) "
              f"{mon_qwk:.3f} | {time.time() - t0:.0f}s | ~{eta_min:.0f} min left")

    outputs = {}
    for set_name, idx in eval_sets.items():                                # final EMA model, with and without TTA
        outputs.update({f"{set_name}_idx": idx, f"{set_name}_y": labels[idx],
                        f"{set_name}_score": predict(ema, idx, cfg),
                        f"{set_name}_score_tta": predict(ema, idx, cfg, tta=True)})
    np.savez_compressed(run_dir / "outputs.npz", **outputs)
    if SAVE_WEIGHTS:
        save_atomically(run_dir / "final_ema.pt", ema.state_dict())
    done_file.write_text(json.dumps({"minutes": state["seconds"] / 60}, indent=2))   # written last = run complete
    ckpt_path.unlink(missing_ok=True)
    return load_run(run_dir, cfg)


def heldout_qwk(run, set_name="val", tta=True):
    # The screening metric: cross-fitted QWK on the held-out images.
    return crossfit_qwk(run[f"{set_name}_score_tta" if tta else f"{set_name}_score"], run[f"{set_name}_y"])


runs = {}
SCREEN_SETS = {"val": SPLIT["val"], "test": TEST}           # val is monitored; test is saved, not looked at

## 8. Block 1: cheap levers (EfficientNet-B3, 300 px, 2 seeds)

R0 and each one-change variant, with 2 training seeds each. A lever is **adopted** when its mean gain in cross-fitted validation QWK (with TTA) is at least `ADOPT_MIN_GAIN` (+0.003). This rule is fixed in advance, and all adopted levers are combined for Block 2.

In [ ]:
block1_names = {}
for lever, change in {"R0": {}, **LEVERS}.items():
    for seed in SCREEN_SEEDS:
        name = f"b1_{lever}_s{seed}"
        block1_names[(lever, seed)] = name
        run = run_experiment(name, replace(BASE_CFG, **change, seed=seed), SPLIT["train"], SCREEN_SETS,
                             allowed=1 in RUN_BLOCKS and (B1_SEEDS is None or seed in B1_SEEDS))
        if run is not None:
            runs[name] = run

block1_done = all(name in runs for name in block1_names.values())
rows = []
for (lever, seed), name in block1_names.items():
    if name in runs:
        r = runs[name]
        rows.append({"lever": lever, "seed": seed, "val_qwk_cf_tta": heldout_qwk(r), "val_qwk_cf": heldout_qwk(r, tta=False),
                     "val_mse_tta": float(np.mean((r["val_score_tta"] - r["val_y"]) ** 2)),
                     "minutes": json.loads((RUNS_DIR / name / "results.json").read_text())["minutes"]})
block1 = pd.DataFrame(rows)
ADOPTED = []
if len(block1):
    table1 = block1.groupby("lever", sort=False).agg(val_qwk_cf_tta=("val_qwk_cf_tta", "mean"), val_qwk_cf=("val_qwk_cf", "mean"),
                                                     val_mse_tta=("val_mse_tta", "mean"), seeds=("seed", "count"),
                                                     minutes=("minutes", "mean"))
    if "R0" in table1.index:
        base_by_seed = block1[block1["lever"] == "R0"].set_index("seed")["val_qwk_cf_tta"]
        paired = block1.assign(gain=block1["val_qwk_cf_tta"] - block1["seed"].map(base_by_seed))
        table1["gain_vs_R0"] = paired.groupby("lever", sort=False)["gain"].mean()
        table1["adopted"] = [lever != "R0" and block1_done and gain >= ADOPT_MIN_GAIN
                             for lever, gain in table1["gain_vs_R0"].items()]
        ADOPTED = [lever for lever, ok in table1["adopted"].items() if ok]
    table1.to_csv(OUT_DIR / "block1_levers.csv")
    display(table1.round(4))
    print(f"TTA gain (mean over Block 1 runs): {(block1['val_qwk_cf_tta'] - block1['val_qwk_cf']).mean():+.4f}")
print("Block 1 complete." if block1_done else "Block 1 not complete yet: Block 2 waits for it.",
      "Adopted levers:", ADOPTED or "none")

## 9. Block 2: resolution and backbone (adopted levers, seed 42)

Each candidate is the combined recipe (R0 + adopted levers) with another backbone or image size. The `TOP_K` candidates with the best cross-fitted validation QWK (TTA) go to Block 3. `b3_300` also checks that the adopted levers add up.

*Expected time per run on a T4: b3_300 ~10 min, b3_384 ~15, b4ns_380 ~30, v2s21k_384 ~30.*

In [ ]:
COMBINED = replace(BASE_CFG, **{k: v for lever in ADOPTED for k, v in LEVERS[lever].items()})
block2_names = {cand: f"b2_{cand}" for cand in CANDIDATES}
TOP = []
if block1_done:
    for cand, change in CANDIDATES.items():
        run = run_experiment(block2_names[cand], replace(COMBINED, **change), SPLIT["train"], SCREEN_SETS,
                             allowed=2 in RUN_BLOCKS and (B2_CANDS is None or cand in B2_CANDS))
        if run is not None:
            runs[block2_names[cand]] = run
block2_done = block1_done and all(name in runs for name in block2_names.values())
rows = [{"candidate": cand, "backbone": CANDIDATES[cand]["backbone"], "image_size": CANDIDATES[cand]["image_size"],
         "val_qwk_cf_tta": heldout_qwk(runs[name]), "val_qwk_cf": heldout_qwk(runs[name], tta=False),
         "minutes": json.loads((RUNS_DIR / name / "results.json").read_text())["minutes"]}
        for cand, name in block2_names.items() if name in runs]
block2 = pd.DataFrame(rows)
if len(block2):
    block2 = block2.sort_values("val_qwk_cf_tta", ascending=False).reset_index(drop=True)
    block2.to_csv(OUT_DIR / "block2_candidates.csv", index=False)
    display(block2.round(4))
if block2_done:
    TOP = block2["candidate"].head(TOP_K).tolist()
print("Combined recipe:", {k: v for k, v in asdict(COMBINED).items() if getattr(BASE_CFG, k) != v} or "R0 unchanged")
print("Block 2 complete. Going to Block 3:" if block2_done else "Block 2 not complete yet: Block 3 waits for it.", TOP)

## 10. Block 3: 5-fold cross-validation ensemble

For each of the `TOP_K` candidates: 5 models, each trained on 4/5 of train + val and predicting the remaining 1/5 (out-of-fold, OOF) and the test set. Cut-offs are later tuned on all 3,112 OOF predictions, which is 5.7x more images than a single validation set.

In [ ]:
cv_names = {(cand, k): f"b3_cv_{cand}_f{k}" for cand in TOP for k in range(CV_FOLDS)}
for (cand, k), name in cv_names.items():
    fit_idx, oof_idx = FOLDS[k]
    run = run_experiment(name, replace(COMBINED, **CANDIDATES[cand]), fit_idx, {"oof": oof_idx, "test": TEST},
                         allowed=3 in RUN_BLOCKS and (B3_TOP is None or cand in [TOP[i] for i in B3_TOP]))
    if run is not None:
        runs[name] = run
CV_DONE = [cand for cand in TOP if all((cand, k) in cv_names and cv_names[(cand, k)] in runs for k in range(CV_FOLDS))]


def cv_scores(cand, tta=True):
    # Out-of-fold scores for TRAINVAL (each image predicted by the model that did not train on it) and the
    # test scores averaged over the folds' models.
    key = "_score_tta" if tta else "_score"
    oof = pd.Series(np.nan, index=np.arange(len(df)))
    tests = []
    for k in range(CV_FOLDS):
        r = runs[cv_names[(cand, k)]]
        oof[r["oof_idx"]] = r["oof" + key]
        tests.append(r["test" + key])
    return oof[TRAINVAL].to_numpy(), np.mean(tests, 0)


Y_TRAINVAL = labels[TRAINVAL]
for cand in CV_DONE:
    print(f"{cand}: OOF QWK (cross-fitted, TTA) {crossfit_qwk(cv_scores(cand)[0], Y_TRAINVAL):.4f} on {len(TRAINVAL)} images")
if len(CV_DONE) < len(TOP) or not TOP:
    print("Block 3 not complete yet:", {cand: sum(cv_names[(cand, k)] in runs for k in range(CV_FOLDS)) for cand in TOP},
          "folds finished")

## 11. Final: the test set, opened once

**Headline (fixed in advance):** the blend (mean) of the Block 3 cross-validation ensembles, with TTA, and cut-offs tuned on the out-of-fold predictions. The other rows put it in context:

* the Stage 4 baseline (A) and hypothesis (D), seed 42;
* the best single Block 2 model (cut-offs tuned on validation);
* each cross-validation ensemble on its own;
* the blend without TTA.

Each row has a 95% bootstrap interval, and QWK on the test images **without** a twin in the training data (leakage removed). Paired bootstrap tests compare the headline with Stage 4.

In [ ]:
final_rows, final_preds, final_cutoffs, headline = [], {}, {}, None
if not OPEN_TEST:
    print("Test set locked in this kernel; it is opened once, in k4-final.")
else:


    def add_row(label, pred, ref_score, note=""):
        lo, hi = bootstrap_qwk(Y_TEST, pred)
        final_preds[label] = pred
        final_rows.append({"model": label, **compute_metrics(Y_TEST, pred, ref_score), "qwk_ci_low": lo, "qwk_ci_high": hi,
                           "qwk_without_twins": qwk(Y_TEST[~TEST_TWIN], pred[~TEST_TWIN]), "note": note})


    ce42, mse42 = stage4_outputs("ce_s42"), stage4_outputs("mse_s42")
    if ce42 is not None and mse42 is not None:
        add_row("Stage 4 A: cross-entropy, vote (seed 42)", majority_vote(ce42["snap_test"]),
                ce42["snap_test"].mean(0)[:, 2:].sum(1), "baseline")
        ens_t, ens_v = mse42["snap_test"].mean(0)[:, 0], mse42["snap_val"].mean(0)[:, 0]
        add_row("Stage 4 D: ordinal, tuned cut-offs (seed 42)", apply_cutoffs(ens_t, fit_cutoffs(ens_v, mse42["y_val"])),
                ens_t, "Stage 4 hypothesis")
    if block2_done:
        best = runs[block2_names[TOP[0]]]
        cut = fit_cutoffs(best["val_score_tta"], best["val_y"])
        final_cutoffs[f"single {TOP[0]}"] = cut.round(2).tolist()
        add_row(f"single model {TOP[0]} (val cut-offs, TTA)", apply_cutoffs(best["test_score_tta"], cut), best["test_score_tta"])
    if CV_DONE:
        blends = {}
        for tta in (True, False):
            oofs, tests = zip(*(cv_scores(cand, tta) for cand in CV_DONE))
            blends[tta] = (np.mean(oofs, 0), np.mean(tests, 0))
            if tta:
                for cand, oof, test in zip(CV_DONE, oofs, tests):
                    cut = fit_cutoffs(oof, Y_TRAINVAL)
                    final_cutoffs[f"cv {cand}"] = cut.round(2).tolist()
                    add_row(f"CV ensemble {cand} (OOF cut-offs, TTA)", apply_cutoffs(test, cut), test,
                            f"OOF QWK {crossfit_qwk(oof, Y_TRAINVAL):.3f}")
        for tta in (False, True):
            oof, test = blends[tta]
            cut = fit_cutoffs(oof, Y_TRAINVAL)
            final_cutoffs[f"blend tta={tta}"] = cut.round(2).tolist()
            add_row(f"blend of {' + '.join(CV_DONE)}" + (" (HEADLINE)" if tta else " without TTA"),
                    apply_cutoffs(test, cut), test, f"OOF QWK {crossfit_qwk(oof, Y_TRAINVAL):.3f}")

    if final_rows:
        final = pd.DataFrame(final_rows)
        final.to_csv(OUT_DIR / "final_test_results.csv", index=False)
        (OUT_DIR / "final_cutoffs.json").write_text(json.dumps(final_cutoffs, indent=2))
        print(f"Test set: {len(TEST)} images ({TEST_TWIN.sum()} with a twin in train + val). Paper (reported): QWK 0.954.")
        display(final[["model", "qwk", "qwk_ci_low", "qwk_ci_high", "qwk_without_twins", "accuracy", "macro_f1",
                       "recall_Severe", "recall_PDR", "off_by_2_or_more", "ref_sensitivity", "ref_specificity",
                       "ref_auc", "note"]].round(3).style.hide(axis="index"))

        headline = next((m for m in final_preds if "HEADLINE" in m), None)
        if headline:
            tests = []
            for other in final_preds:
                if other != headline:
                    lo, hi = bootstrap_qwk(Y_TEST, final_preds[other], final_preds[headline])
                    tests.append({"comparison": f"headline - {other}",
                                  "delta_qwk": qwk(Y_TEST, final_preds[headline]) - qwk(Y_TEST, final_preds[other]),
                                  "ci_low": lo, "ci_high": hi, "excludes_0": bool(lo > 0 or hi < 0)})
            paired = pd.DataFrame(tests)
            paired.to_csv(OUT_DIR / "final_paired_bootstrap.csv", index=False)
            display(paired.round(4).style.hide(axis="index"))
    else:
        print("Nothing to report yet: run the blocks first.")

## 12. Figures

In [ ]:
if len(block1) and "gain_vs_R0" in table1:
    gains = table1["gain_vs_R0"].drop("R0", errors="ignore")
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(gains.index, gains.values, color=["#54A24B" if g >= ADOPT_MIN_GAIN else "#BAB0AC" for g in gains.values])
    ax.axvline(ADOPT_MIN_GAIN, color="k", ls="--", lw=1, label=f"adoption threshold (+{ADOPT_MIN_GAIN})")
    ax.axvline(0, color="k", lw=0.5)
    ax.set(xlabel="gain in cross-fitted val QWK vs. R0 (mean of seeds, TTA)", title="Block 1: one change at a time")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.savefig(FIG_DIR / "block1_levers.png", dpi=150); plt.show()

if len(block2):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(block2["candidate"][::-1], block2["val_qwk_cf_tta"][::-1], color="#4C78A8")
    ax.set(xlabel="cross-fitted val QWK (TTA)", title="Block 2: resolution and backbone",
           xlim=(max(0, block2["val_qwk_cf_tta"].min() - 0.03), min(1, block2["val_qwk_cf_tta"].max() + 0.01)))
    plt.tight_layout(); plt.savefig(FIG_DIR / "block2_candidates.png", dpi=150); plt.show()

if final_rows and headline:
    cm = confusion_matrix(Y_TEST, final_preds[headline], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4.4))
    ax.imshow(cm / cm.sum(1, keepdims=True).clip(1), cmap="Blues", vmin=0, vmax=1)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, cm[r, c], ha="center", va="center", color="white" if cm[r, c] > cm[r].sum() / 2 else "black")
    ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES), xlabel="predicted", ylabel="true",
           title=f"headline: test QWK {qwk(Y_TEST, final_preds[headline]):.3f}")
    ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticklabels(CLASS_NAMES)
    plt.tight_layout(); plt.savefig(FIG_DIR / "final_confusion.png", dpi=150); plt.show()

## 13. Writing it up

* **Part 0** goes in the Discussion. It shows how far evaluation choices alone move the score, and it is the fairest way to talk about the gap to 0.954.
* **Block 1 and Block 2 tables:** a "what helps and by how much" table for the paper, with predictions next to outcomes from `stage4b_plan.md`.
* **Final table:** the headline row, as fixed in advance, with its interval and its leakage-free QWK. If the paper's 0.954 lies inside our interval, say so; if our point estimate passes it, say so too. Either way, also report the interval and the leakage-free number.
* **Limitations:** a single split and test set; one seed for Block 2; Noisy Student and ImageNet-21k weights are not plain ImageNet-1k; selection on 550 validation images; no external data or validation. No clinical claims.

**Files in `MyDrive/dr-grading/stage4b/`:** `protocol_gap.csv`, `block1_levers.csv`, `block2_candidates.csv`, `final_test_results.csv`, `final_paired_bootstrap.csv`, `final_cutoffs.json`, and `figures/`. Download them (not `runs/`) and commit them to `dr-repro/results/stage4b/`.